Data Ingestion and VectorDB

In [3]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import  uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity
from pathlib import Path
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [4]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 5 PDF files to process

Processing: Dell Inspiron 15 Owner's Manual.pdf
  ✓ Loaded 95 pages

Processing: HP PAVILION 17 MAINTENANCE AND SERVICE.pdf
  ✓ Loaded 173 pages

Processing: Laptop Buying Guide 2025-26.pdf
  ✓ Loaded 13 pages

Processing: Lenovo IdeaPad Slim 3 Series Manual.pdf
  ✓ Loaded 42 pages

Processing: Win11QuckGuide.pdf
  ✓ Loaded 3 pages

Total documents loaded: 326


In [5]:
all_pdf_documents

[Document(metadata={'producer': 'Qt 4.8.7', 'creator': 'wkhtmltopdf 0.12.6', 'creationdate': '2026-02-05T22:43:33+01:00', 'title': 'https://www.manualslib.com/manual/705952/Dell-Inspiron-15.html', 'source': "..\\data\\PDF\\Dell Inspiron 15 Owner's Manual.pdf", 'total_pages': 95, 'page': 0, 'page_label': '1', 'source_file': "Dell Inspiron 15 Owner's Manual.pdf", 'file_type': 'pdf'}, page_content="Manualslib.com\n - The Global Manuals Library\nManuals\n \n/ \nBrands\n \n/ \nDell Manuals\n \n/ \nLaptop\n \n/ \nInspiron 15\n \n/ \nOwner's manual\n \n/ \nPDF\nDELL INSPIRON 15 OWNER'S MANUAL\nQuick Links\nBefore You Begin\nBefore Working Inside Your Computer\nRemoving the Base Cover\nRemoving the Battery\nRemoving the Memory Module(S)"),
 Document(metadata={'producer': 'Qt 4.8.7', 'creator': 'wkhtmltopdf 0.12.6', 'creationdate': '2026-02-05T22:43:33+01:00', 'title': 'https://www.manualslib.com/manual/705952/Dell-Inspiron-15.html', 'source': "..\\data\\PDF\\Dell Inspiron 15 Owner's Manual.pdf

In [6]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [7]:
chunks=split_documents(all_pdf_documents)
chunks

Split 326 documents into 684 chunks

Example chunk:
Content: Manualslib.com
 - The Global Manuals Library
Manuals
 
/ 
Brands
 
/ 
Dell Manuals
 
/ 
Laptop
 
/ 
Inspiron 15
 
/ 
Owner's manual
 
/ 
PDF
DELL INSPIRON 15 OWNER'S MANUAL
Quick Links
Before You Begi...
Metadata: {'producer': 'Qt 4.8.7', 'creator': 'wkhtmltopdf 0.12.6', 'creationdate': '2026-02-05T22:43:33+01:00', 'title': 'https://www.manualslib.com/manual/705952/Dell-Inspiron-15.html', 'source': "..\\data\\PDF\\Dell Inspiron 15 Owner's Manual.pdf", 'total_pages': 95, 'page': 0, 'page_label': '1', 'source_file': "Dell Inspiron 15 Owner's Manual.pdf", 'file_type': 'pdf'}


[Document(metadata={'producer': 'Qt 4.8.7', 'creator': 'wkhtmltopdf 0.12.6', 'creationdate': '2026-02-05T22:43:33+01:00', 'title': 'https://www.manualslib.com/manual/705952/Dell-Inspiron-15.html', 'source': "..\\data\\PDF\\Dell Inspiron 15 Owner's Manual.pdf", 'total_pages': 95, 'page': 0, 'page_label': '1', 'source_file': "Dell Inspiron 15 Owner's Manual.pdf", 'file_type': 'pdf'}, page_content="Manualslib.com\n - The Global Manuals Library\nManuals\n \n/ \nBrands\n \n/ \nDell Manuals\n \n/ \nLaptop\n \n/ \nInspiron 15\n \n/ \nOwner's manual\n \n/ \nPDF\nDELL INSPIRON 15 OWNER'S MANUAL\nQuick Links\nBefore You Begin\nBefore Working Inside Your Computer\nRemoving the Base Cover\nRemoving the Battery\nRemoving the Memory Module(S)"),
 Document(metadata={'producer': 'Qt 4.8.7', 'creator': 'wkhtmltopdf 0.12.6', 'creationdate': '2026-02-05T22:43:33+01:00', 'title': 'https://www.manualslib.com/manual/705952/Dell-Inspiron-15.html', 'source': "..\\data\\PDF\\Dell Inspiron 15 Owner's Manual.pdf

Embedding & VectorStoreDB


In [9]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [10]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully. Embedding dimension: 384


VectorStore

In [11]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 684


In [12]:
chunks

[Document(metadata={'producer': 'Qt 4.8.7', 'creator': 'wkhtmltopdf 0.12.6', 'creationdate': '2026-02-05T22:43:33+01:00', 'title': 'https://www.manualslib.com/manual/705952/Dell-Inspiron-15.html', 'source': "..\\data\\PDF\\Dell Inspiron 15 Owner's Manual.pdf", 'total_pages': 95, 'page': 0, 'page_label': '1', 'source_file': "Dell Inspiron 15 Owner's Manual.pdf", 'file_type': 'pdf'}, page_content="Manualslib.com\n - The Global Manuals Library\nManuals\n \n/ \nBrands\n \n/ \nDell Manuals\n \n/ \nLaptop\n \n/ \nInspiron 15\n \n/ \nOwner's manual\n \n/ \nPDF\nDELL INSPIRON 15 OWNER'S MANUAL\nQuick Links\nBefore You Begin\nBefore Working Inside Your Computer\nRemoving the Base Cover\nRemoving the Battery\nRemoving the Memory Module(S)"),
 Document(metadata={'producer': 'Qt 4.8.7', 'creator': 'wkhtmltopdf 0.12.6', 'creationdate': '2026-02-05T22:43:33+01:00', 'title': 'https://www.manualslib.com/manual/705952/Dell-Inspiron-15.html', 'source': "..\\data\\PDF\\Dell Inspiron 15 Owner's Manual.pdf

In [13]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 684 texts...


Batches:   0%|          | 0/22 [00:00<?, ?it/s]

Generated embeddings with shape: (684, 384)
Adding 684 documents to vector store...
Successfully added 684 documents to vector store
Total documents in collection: 1368


Retrieve Pipeline From VectorStore

In [14]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [15]:
rag_retriever

In [16]:
rag_retriever.retrieve("What is the process of replacing a laptop battery?")

Retrieving documents for query: 'What is the process of replacing a laptop battery?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_d58134de_365',
  'content': 'Battery\nDescription Spare part number\n4-cell, 41 WHr 2.8AH Li-ion battery 756743-001\nBefore removing the battery, follow these steps:\n1. Shut down the computer. If you are unsure whether the computer is off or in Hibernation, turn\nthe computer on, and then shut it down through the operating system.\n2. Disconnect all external devices connected to the computer.\n3. Disconnect the power from the computer by first unplugging the power cord from the AC outlet\nand then unplugging the AC adapter from the computer.\nRemove the battery:\n1. Turn the computer upside down on a flat surface.\n2. Slide the battery lock latch (1) and then slide the battery release latch (2) to release the battery.\nNOTE: The battery release latch automatically returns to its original position.\n3. Pivot the battery upward (3), and then remove the battery from the computer (4).\nReverse this procedure to install the battery.\n76 Chapter 5   Removal and replac ement pro

RAG Pipeline- VectorDB To LLM Output Generation

In [29]:
import os
from dotenv import load_dotenv
load_dotenv()

print(os.getenv("YOUR_GROQ_API_KEY"))

None


In [18]:
# 1. Groq specific integration
from langchain_groq import ChatGroq

# 2. Core Prompting (where PromptTemplate actually lives)
from langchain_core.prompts import PromptTemplate

# 3. Core Messages (FIXED: Moved from community to core.messages)
from langchain_core.messages import HumanMessage, SystemMessage

print("✅ All imports are now correct!")

✅ All imports are now correct!


In [19]:
class GroqLLM:
    def __init__(self, model_name: str = "llama-3.1-8b-instant", api_key: str =None):
        """
        Initialize Groq LLM
        
        Args:
            model_name: Groq model name (qwen2-72b-instruct, llama3-70b-8192, etc.)
            api_key: Groq API key (or set GROQ_API_KEY environment variable)
        """
        self.model_name = model_name
        self.api_key = api_key or os.environ.get("GROQ_API_KEY")
        
        if not self.api_key:
            raise ValueError("Groq API key is required. Set GROQ_API_KEY environment variable or pass api_key parameter.")
        
        self.llm = ChatGroq(
            groq_api_key=self.api_key,
            model_name=self.model_name,
            temperature=0.1,
            max_tokens=1024
        )
        
        print(f"Initialized Groq LLM with model: {self.model_name}")

    def generate_response(self, query: str, context: str, max_length: int = 500) -> str:
        """
        Generate response using retrieved context
        
        Args:
            query: User question
            context: Retrieved document context
            max_length: Maximum response length
            
        Returns:
            Generated response string
        """
        
        # Create prompt template
        prompt_template = PromptTemplate(
            input_variables=["context", "question"],
            template="""You are a helpful AI assistant. Use the following context to answer the question accurately and concisely.

Context:
{context}

Question: {question}

Answer: Provide a clear and informative answer based on the context above. If the context doesn't contain enough information to answer the question, say so."""
        )
        
        # Format the prompt
        formatted_prompt = prompt_template.format(context=context, question=query)
        
        try:
            # Generate response
            messages = [HumanMessage(content=formatted_prompt)]
            response = self.llm.invoke(messages)
            return response.content
            
        except Exception as e:
            return f"Error generating response: {str(e)}"
        
    def generate_response_simple(self, query: str, context: str) -> str:
        """
        Simple response generation without complex prompting
        
        Args:
            query: User question
            context: Retrieved context
            
        Returns:
            Generated response
        """
        simple_prompt = f"""Based on this context: {context}

Question: {query}

Answer:"""
        
        try:
            messages = [HumanMessage(content=simple_prompt)]
            response = self.llm.invoke(messages)
            return response.content
        except Exception as e:
            return f"Error: {str(e)}"

In [20]:
# Initialize Groq LLM (you'll need to set GROQ_API_KEY environment variable)
try:
    groq_llm = GroqLLM()
    print("Groq LLM initialized successfully!")
except ValueError as e:
    print(f"Warning: {e}")
    print("Please set your GROQ_API_KEY environment variable to use the LLM.")
    groq_llm = None

Initialized Groq LLM with model: llama-3.1-8b-instant
Groq LLM initialized successfully!


In [21]:
### get the context from the retriever and pass it to the LLM

rag_retriever.retrieve("What is the process of replacing a laptop battery?")

Retrieving documents for query: 'What is the process of replacing a laptop battery?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_d58134de_365',
  'content': 'Battery\nDescription Spare part number\n4-cell, 41 WHr 2.8AH Li-ion battery 756743-001\nBefore removing the battery, follow these steps:\n1. Shut down the computer. If you are unsure whether the computer is off or in Hibernation, turn\nthe computer on, and then shut it down through the operating system.\n2. Disconnect all external devices connected to the computer.\n3. Disconnect the power from the computer by first unplugging the power cord from the AC outlet\nand then unplugging the AC adapter from the computer.\nRemove the battery:\n1. Turn the computer upside down on a flat surface.\n2. Slide the battery lock latch (1) and then slide the battery release latch (2) to release the battery.\nNOTE: The battery release latch automatically returns to its original position.\n3. Pivot the battery upward (3), and then remove the battery from the computer (4).\nReverse this procedure to install the battery.\n76 Chapter 5   Removal and replac ement pro

Integration VectorDBContext pipeline with LLM output

In [22]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="llama-3.1-8b-instant",temperature=0.1,max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [23]:
answer=rag_simple("What is the process of replacing a keyboard of dell laptop?",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'What is the process of replacing a keyboard of dell laptop?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
To replace a keyboard on a Dell laptop, follow these steps:

1. Slide the keyboard and keyboard-backlight cables into the respective connectors and press down the latches to secure the cables.
2. Turn the keyboard over, slide the tabs on the keyboard into the slots on the palm-rest assembly, and snap the keyboard into place.

Note: Before working inside your computer, read the safety information and follow the steps in "Before Working Inside Your Computer" and "After Working Inside Your Computer".


Enhanced RAG Pipeline Features

In [24]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("Hard Negative Mining Technqiues", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])


Retrieving documents for query: 'Hard Negative Mining Technqiues'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)
Answer: No relevant context found.
Sources: []
Confidence: 0.0
Context Preview: 


In [25]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("what is attention is all you need", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'what is attention is all you need'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)

Final Answer: No relevant context found.
Summary: There is no information to summarize as the provided answer does not contain any context or details.
History: {'question': 'what is attention is all you need', 'answer': 'No relevant context found.', 'sources': [], 'summary': 'There is no information to summarize as the provided answer does not contain any context or details.'}


In [26]:
### Save Vector Database to Pickle File for Use in Other Applications

import pickle

# Prepare data to pickle
vector_db_data = {
    'documents': chunks,  # List of LangChain documents
    'embeddings': embeddings,  # Numpy array of embeddings
    'embedding_model_name': 'all-MiniLM-L6-v2'  # Assuming this is the model used
}

# Save to pickle file
with open('../data/vector_db.pkl', 'wb') as f:
    pickle.dump(vector_db_data, f)

print("Vector database saved to '../data/vector_db.pkl'")

### Code to Use in Another Application

print("""
# Copy and paste this code into your other application:

import pickle
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Load the pickled vector database
with open('path/to/vector_db.pkl', 'rb') as f:
    vector_db_data = pickle.load(f)

documents = vector_db_data['documents']
embeddings = vector_db_data['embeddings']
embedding_model_name = vector_db_data['embedding_model_name']

# Initialize the embedding model
model = SentenceTransformer(embedding_model_name)

# Simple retriever class
class PickleVectorRetriever:
    def __init__(self, documents, embeddings, model):
        self.documents = documents
        self.embeddings = embeddings
        self.model = model

    def retrieve(self, query, top_k=5):
        # Generate query embedding
        query_embedding = self.model.encode([query])[0]

        # Calculate similarities
        similarities = cosine_similarity([query_embedding], self.embeddings)[0]

        # Get top k indices
        top_indices = np.argsort(similarities)[-top_k:][::-1]

        # Prepare results
        results = []
        for idx in top_indices:
            results.append({
                'content': self.documents[idx].page_content,
                'metadata': self.documents[idx].metadata,
                'similarity_score': similarities[idx]
            })

        return results

# Initialize retriever
retriever = PickleVectorRetriever(documents, embeddings, model)

# Example usage
query = "Your question here"
results = retriever.retrieve(query, top_k=3)
for result in results:
    print(f"Score: {result['similarity_score']:.3f}")
    print(f"Content: {result['content'][:200]}...")
    print("---")

# You can integrate this with any LLM for RAG
""")

Vector database saved to '../data/vector_db.pkl'

# Copy and paste this code into your other application:

import pickle
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Load the pickled vector database
with open('path/to/vector_db.pkl', 'rb') as f:
    vector_db_data = pickle.load(f)

documents = vector_db_data['documents']
embeddings = vector_db_data['embeddings']
embedding_model_name = vector_db_data['embedding_model_name']

# Initialize the embedding model
model = SentenceTransformer(embedding_model_name)

# Simple retriever class
class PickleVectorRetriever:
    def __init__(self, documents, embeddings, model):
        self.documents = documents
        self.embeddings = embeddings
        self.model = model

    def retrieve(self, query, top_k=5):
        # Generate query embedding
        query_embedding = self.model.encode([query])[0]

        # Calculate similarities
        similarities = cosine